# German Podcast to English Podcast (Runpod, Hugging Face only)

This notebook runs end-to-end on a Runpod instance with 1 RTX 5090 (32GB VRAM), 31GB RAM, and 12 CPUs:
1. Read a German MP3 podcast
2. Transcribe with diarisation
3. Translate to English
4. Generate a multi-speaker English podcast
5. Save the translated MP3


## Model selection (for RTX 5090 and <60 minutes target)

- **ASR**: `primeline/whisper-large-v3-turbo-german` (German-finetuned Whisper Turbo; 809M params; 2.628% WER avg on CV19/MLS/Tuda-De; loaded via transformers `pipeline` — WhisperX is not used because it requires CTranslate2-format weights (`model.bin`) which this model does not have)
- **Diarisation**: `pyannote/speaker-diarization-community-1` via pyannote.audio 4.0.4. Audio passed as a pre-loaded waveform tensor to bypass `AudioDecoder` (torchcodec), which is incompatible with PyTorch 2.8 stable.
- **Translation**: `google/translategemma-12b-it` (TranslateGemma — purpose-built translation model released Jan 2026; outperforms NLLB-200 and even Gemma 3 27B on COMET/MetricX; handles conversational context; fits RTX 5090 at FP16 ~22 GB)
- **TTS**: `SWivid/F5-TTS` (flow-matching TTS with zero-shot voice cloning; Apache 2.0; ~6 GB VRAM; RTF 0.15; replaces XTTS-v2 whose parent company Coqui AI shut down Jan 2024)

### Alternatives considered

| Stage | Runner-up | Notes |
|-------|-----------|-------|
| ASR | `Qwen/Qwen3-ASR-1.7B` | SOTA accuracy (Jan 2026), beats whisper-large-v3 across benchmarks, built-in forced aligner, only ~5 GB VRAM. Requires new pipeline code. |
| ASR | `nvidia/canary-1b-v2` | 25 European languages, 749 RTFx, CC-BY-4.0. Requires NeMo framework. |
| Diarisation | `BUT-FIT/diarizen-wavlm-large-s80-md-v2` | Lower DER than pyannote on several benchmarks (e.g. AMI-SDM 13.9% vs 19.9%), but CC-BY-NC license and custom pipeline. |
| Translation | `ByteDance-Seed/Seed-X-PPO-7B` | Competes with GPT-4o quality at 7B; OpenMDW (MIT-like) license; only ~14 GB VRAM. |
| Translation | `Unbabel/Tower-Plus-9B` | Best document-level translation (good for podcasts); CC-BY-NC-SA license (non-commercial). |
| TTS | `ResembleAI/chatterbox` | MIT license, beats ElevenLabs in blind tests, emotion control. |
| TTS | `nari-labs/Dia-1.6B` | Apache 2.0, purpose-built for multi-speaker dialogue with `[S1]`/`[S2]` tags. English only. |
| TTS | `FunAudioLLM/CosyVoice2-0.5B` | Apache 2.0, SOTA speaker similarity (78%), 150ms streaming latency. |

**Do NOT install torchcodec.** No wheel is compatible with PyTorch 2.8.0 stable (see LEARNINGS.md §2). Audio is pre-loaded via librosa instead.

Before running: accept model terms on Hugging Face for `pyannote/speaker-diarization-community-1` and set your read token in `HF_TOKEN`. Place your German MP3 at `INPUT_AUDIO_PATH`.


In [ ]:
%pip -q install "pyannote.audio==4.0.4" transformers accelerate sentencepiece "f5-tts==1.1.16" "pydub==0.25.1" soundfile librosa


In [ ]:
import os
import torch
import numpy as np
import librosa
import pandas as pd
from pathlib import Path
from pydub import AudioSegment
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline as hf_pipeline
from pyannote.audio import Pipeline as DiarizationPipeline
from f5_tts.api import F5TTS

HF_TOKEN = os.environ["HF_TOKEN"]
INPUT_AUDIO_PATH = "german_podcast.mp3"
OUTPUT_AUDIO_PATH = "translated_podcast_en.mp3"
audio_path = INPUT_AUDIO_PATH
device = "cuda"


In [ ]:
SAMPLE_RATE = 16000

# Pre-load audio once as a numpy array (16 kHz mono) for both ASR and diarization.
# Passing a pre-loaded array avoids any file-path code path in transformers/pyannote,
# bypassing torchcodec entirely (torchcodec has no working wheel for PyTorch 2.8 stable).
audio_array, _ = librosa.load(audio_path, sr=SAMPLE_RATE, mono=True)

# --- ASR ---
# WhisperX is not used: it requires CTranslate2-format weights (model.bin) but
# primeline/whisper-large-v3-turbo-german ships as .safetensors. Use transformers directly.
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
asr_model_id = "primeline/whisper-large-v3-turbo-german"

asr_model = AutoModelForSpeechSeq2Seq.from_pretrained(
    asr_model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
asr_model.to(device)
processor = AutoProcessor.from_pretrained(asr_model_id)

asr_pipe = hf_pipeline(
    "automatic-speech-recognition",
    model=asr_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    max_new_tokens=128,
    chunk_length_s=30,
    batch_size=16,
    return_timestamps=True,
    torch_dtype=torch_dtype,
    device=device,
)

asr_result = asr_pipe(
    {"raw": audio_array, "sampling_rate": SAMPLE_RATE},
    generate_kwargs={"language": "german"},
)
asr_chunks = asr_result["chunks"]  # list of {"text": ..., "timestamp": (start, end)}

# --- Diarization ---
# pyannote 4.x: use token= (not use_auth_token= which was removed in 4.0)
diarize_pipeline = DiarizationPipeline.from_pretrained(
    "pyannote/speaker-diarization-community-1",
    token=HF_TOKEN,
)
diarize_pipeline.to(torch.device(device))

# Pass pre-loaded waveform tensor — bypasses pyannote's AudioDecoder / torchcodec
waveform_tensor = torch.tensor(audio_array).unsqueeze(0)  # (1, samples)
diarization = diarize_pipeline({"waveform": waveform_tensor, "sample_rate": SAMPLE_RATE})

# --- Assign speakers to ASR chunks by maximum overlap ---
def get_speaker(start, end, diarization):
    overlap = {}
    # pyannote 4.x: DiarizeOutput.speaker_diarization yields (turn, speaker) 2-tuples
    # pyannote ≤3.x used diarization.itertracks(yield_label=True) → (turn, _, speaker)
    for turn, speaker in diarization.speaker_diarization:
        o = min(turn.end, end) - max(turn.start, start)
        if o > 0:
            overlap[speaker] = overlap.get(speaker, 0) + o
    return max(overlap, key=overlap.get) if overlap else "SPEAKER_00"

rows = []
for chunk in asr_chunks:
    start, end = chunk["timestamp"]
    if end is None:
        end = start + 30.0
    speaker = get_speaker(start, end, diarization)
    rows.append({"start": start, "end": end, "speaker": speaker, "text": chunk["text"].strip()})

segments_df = pd.DataFrame(rows)
segments_df["speaker"] = segments_df["speaker"].fillna("SPEAKER_00")


In [ ]:
model_id = "google/translategemma-12b-it"
tokenizer = AutoTokenizer.from_pretrained(model_id, token=HF_TOKEN)
translate_model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.float16, device_map="auto", token=HF_TOKEN
)

translated_texts = []
for text in segments_df["text"].tolist():
    # translategemma-12b-it is instruction-tuned — must use the chat template.
    # Passing a raw string skips <start_of_turn> tokens, producing garbage logits
    # with NaN probabilities that crash torch.multinomial with AcceleratorError.
    messages = [{"role": "user", "content": f"Translate the following German text to English.\nGerman: {text}\nEnglish:"}]
    inputs = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True, add_generation_prompt=True
    ).to("cuda")  # with device_map="auto", use "cuda" not translate_model.device
    with torch.no_grad():
        # do_sample=False (greedy) is deterministic and avoids multinomial entirely
        outputs = translate_model.generate(**inputs, max_new_tokens=512, do_sample=False)
    decoded = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    translated_texts.append(decoded.strip())

segments_df["text_en"] = translated_texts
segments_df.to_csv("podcast_transcript_en.csv", index=False)

del translate_model, tokenizer
torch.cuda.empty_cache()


In [ ]:
work_dir = Path("translated_segments")
work_dir.mkdir(exist_ok=True)
source_audio = AudioSegment.from_file(audio_path)
speaker_refs = segments_df.sort_values("start").drop_duplicates("speaker")[["speaker", "start", "end"]]
speaker_refs["ref_path"] = [work_dir / f"{speaker}_ref.wav" for speaker in speaker_refs["speaker"]]
for row in speaker_refs.itertuples(index=False):
    source_audio[int(row.start * 1000):int(row.end * 1000)].export(row.ref_path, format="wav")
speaker_ref_map = dict(zip(speaker_refs["speaker"], speaker_refs["ref_path"]))

f5tts = F5TTS(model="F5TTS_v1_Base")

segment_paths = []
for row in segments_df.itertuples(index=False):
    segment_path = work_dir / f"seg_{int(row.start * 1000):09d}.wav"
    ref_path = str(speaker_ref_map[row.speaker])
    ref_text = f5tts.transcribe(ref_audio=ref_path)
    f5tts.infer(
        ref_file=ref_path,
        ref_text=ref_text,
        gen_text=row.text_en,
        file_wave=str(segment_path),
    )
    segment_paths.append(segment_path)

In [ ]:
translated_audio = AudioSegment.silent(duration=0)
cursor_ms = 0
for row, segment_path in zip(segments_df.itertuples(index=False), segment_paths):
    start_ms = int(row.start * 1000)
    translated_audio = translated_audio + AudioSegment.silent(duration=max(start_ms - cursor_ms, 0))
    segment_audio = AudioSegment.from_wav(segment_path)
    translated_audio = translated_audio + segment_audio
    cursor_ms = len(translated_audio)
translated_audio.export(OUTPUT_AUDIO_PATH, format="mp3")
